In [59]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

from itertools import product
import os
import h5py
from tqdm import tqdm
import pandas as pd

#few utils
from few.utils.utility import get_p_at_t
from few.utils.constants import MTSUN_SI
from few.utils.geodesic import get_fundamental_frequencies
#few trajectory
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import SuperKludgeFlux
#few waveform
from few.waveform import FastKerrEccentricEquatorialFlux, GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from few.utils.constants import YRSID_SI

#sef imports
from stableemrifisher.fisher import StableEMRIFisher
from stableemrifisher.utils import generate_PSD, padding, inner_product
from stableemrifisher.fisher.derivatives import derivative
from stableemrifisher.fisher.stablederivative import StableEMRIDerivative
from stableemrifisher.noise import sensitivity_LWA

#lisa-on-gpu import
from fastlisaresponse import ResponseWrapper  # Response function 

#LISAanalysistools imports
from lisatools.detector import ESAOrbits, EqualArmlengthOrbits #ESAOrbits correspond to esa-trailing-orbits.h5, EqualArmlengthOrbits are equalarmlength-orbits.h5
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from lisatools.sensitivity import get_sensitivity,CornishLISASens

use_gpu = True
from parismc.sampler import SamplerConfig
from parismc.sampler import Sampler

from smt.sampling_methods import LHS

if not use_gpu:
    
    import few
    
    #tune few configuration
    cfg_set = few.get_config_setter(reset=True)
    
    cfg_set.enable_backends("cpu")
    cfg_set.set_log_level("info");
else:
    pass #let the backend decide for itself

In [60]:
#waveform class setup
waveform_class = SuperKludgeWaveform
max_step_days = 10.0 #max trajectory step size in days
inspiral_kwargs = {
    "err":1e-11, #default = 1e-11
    "max_step_size":max_step_days*24*60*60, #in seconds
    "use_gpu":use_gpu
}
sum_kwargs = {
    "pad_output": True, # True if expecting waveforms smaller than LISA observation window.
  #  "use_gpu":use_gpu
}

waveform_class_kwargs = dict(inspiral_kwargs=inspiral_kwargs,
                              mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                              sum_kwargs=sum_kwargs,
                              use_gpu=use_gpu)

waveform_generator = GenerateEMRIWaveform
waveform_generator_kwargs = dict(return_list=False)


In [61]:
if(use_gpu):
    xp=cp
else:
    xp=np

m1 = 1e6
m2 = 10
a = 0.8 # 0.95
e0 = 0.4 # 0.6 just spin first
xI0 = 1.0
dist = 0.4
qS = xp.pi/4
phiS = 1.0
qK = 1 
phiK = xp.pi/3
Phi_phi0 = 0.9
Phi_theta0 =0.5
Phi_r0 = 0.4

dt = 10.0
T = 0.5

chi2 = 0.0

dev_0_p=0.0
dev_0_e=0.0
dev_1_p=0.0
dev_1_e=0.0
dev_2_p=0.0
dev_2_e=0.0
evolve_1PA = False
evolve_primary = False
evolve_2PA = False
deviation_included=True
p0=7.5

print(use_gpu)
pars_list_com = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,\
             chi2,evolve_1PA,evolve_primary,evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args={"chi2":chi2,"evolve_1PA":evolve_1PA,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,"deviation_included":deviation_included,"dev0p":dev_0_p,
"dev0e":dev_0_e,"dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

emri_kwargs = {"T":T, "dt":dt}

param_names_com = ['m1','m2','a','p0','e0','xI0','dist','qS','phiS','qK','phiK','Phi_phi0','Phi_theta0','Phi_r0',"chi2",
               "evolve_1PA","evolve_primary","evolve_2PA","deviation_included","dev0p","dev0e","dev1p","dev1e","dev2p","dev2e"]

True


In [62]:
der_order = 8
Ndelta=10
sef = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                       stats_for_nerds = True, use_gpu = use_gpu,
                       deriv_type='stable',
                       noise_model=get_sensitivity,
                       noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                       channels=["A","E"],
                       T = T, dt = dt,
                       stability_plot = False,
                       der_order = der_order, Ndelta = Ndelta,
                       plunge_check=True, return_derivatives=False
                       )
                   

SNR = sef.SNRcalc_SEF(*pars_list_com,**emri_kwargs,use_gpu=use_gpu)
print("SNR: ", SNR)

KeyboardInterrupt: 

In [ ]:
#initialize the 0PA (approximate) model
evolve_1PA = False 
evolve_primary = False
evolve_2PA = False #for approximate model
deviation_included=True
add_param_args={"chi2":chi2,"evolve_1PA":evolve_1PA,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,"deviation_included":deviation_included,"dev0p":dev_0_p,
"dev0e":dev_0_e,"dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}


param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']
pars_list = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]

param_dict = {
    'm1': m1,
    'm2': m2,
    'a': a,
    'p0': p0,
    'e0': e0,
    'xI0': xI0,
    'dist': dist,
    'qS': qS,
    'phiS': phiS,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0
}


Fisher = sef(wave_params = param_dict,param_names=param_names, add_param_args=add_param_args,
            live_dangerously = False, stability_plot = True,der_order = der_order, Ndelta = Ndelta,
            )


In [ ]:
def logmasstransform(Fisher, m1, index_of_m1 = 0):    
    J = np.eye(len(Fisher))
    J[index_of_m1,index_of_m1] = m1
    
    return J.T@Fisher@J
fisher_=logmasstransform(Fisher, m1, index_of_m1 = 0)
cov=np.linalg.inv(fisher_)
std= np.sqrt(np.diag(cov))
params_truth_in = np.array([np.log(m1), m2, a, p0, e0, qS, phiS, Phi_phi0,Phi_r0,dev_0_p,dev_0_e])

In [ ]:
chi2=0
deviation_included=True
evolve_1PA=True
evolve_primary=False
evolve_2PA=False
use_gpu=True
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
superkludge_wave = GenerateEMRIWaveform(SuperKludgeWaveform,\
                                    sum_kwargs=sum_kwargs,\
                                    return_list=True,
                                    mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                                    inspiral_kwargs=inspiral_kwargs,
                                    use_gpu=use_gpu)
print(add_args)

waveform_true = superkludge_wave(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0, *add_args, dt=dt, T=T)
PSD=generate_PSD(waveform_true,dt,use_gpu=use_gpu,
                noise_PSD=get_sensitivity,
                noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                channels=["A","E"])
waveform_true=xp.array(waveform_true)
waveform_true

In [ ]:
def inner_prod_without_phase(a, b, PSD, dt, window=None, fmin=None, fmax=None, use_gpu=False):

    if use_gpu:
        xp = cp
    else:
        xp = np

    # print("fmin: {}, fmax: {}".format(fmin, fmax))

    # frequency cutoff mask
    if (fmin != None) or (fmax != None):

        length = len(a[0])
        freq = xp.fft.rfftfreq(length) / dt

        if use_gpu:
            freq = freq.get()  # convert to numpy

        if fmin != None:
            mask_min = freq > fmin

        if fmax != None:
            mask_max = freq < fmax

        if (fmin != None) and (fmax == None):
            freq_mask = mask_min
        elif (fmin == None) and (fmax != None):
            freq_mask = mask_max
        else:
            freq_mask = xp.logical_and(mask_min, mask_max)

    else:
        length = len(a[0])
        
        freq = xp.fft.rfftfreq(length) / dt

        freq_mask = np.full(len(freq), True, dtype=bool)

    freq_mask = freq_mask[1:]  # skip the first element corresponding to f = 0.0

    a = xp.atleast_2d(a)
    b = xp.atleast_2d(b)
    PSD = xp.atleast_2d(
        xp.asarray(PSD)
    )  # handle passing the same PSD for multiple channels

    N = a.shape[1]

    df = (N * dt) ** -1

    if window is not None:
        window = xp.atleast_2d(xp.asarray(window))
        a_in = a * window
        b_in = b * window
    else:
        a_in, b_in = a, b

    if xp.iscomplexobj(a_in):
        a_fft_plus = (dt * xp.fft.rfft(a_in.real, axis=-1)[:, 1:])[:, freq_mask]
        a_fft_cross = (dt * xp.fft.rfft(a_in.imag, axis=-1)[:, 1:])[:, freq_mask]

        b_fft_plus = (dt * xp.fft.rfft(b_in.real, axis=-1)[:, 1:])[:, freq_mask]
        b_fft_cross = (dt * xp.fft.rfft(b_in.imag, axis=-1)[:, 1:])[:, freq_mask]

        inner_prod = (
            4
            * df
            * (
                xp.abs(a_fft_plus.conj() * b_fft_plus + a_fft_cross * b_fft_cross.conj()
                / PSD[:, freq_mask])
            ).sum()
        )

    else:
        a_fft = (dt * xp.fft.rfft(a_in, axis=-1)[:, 1:])[:, freq_mask]
        b_fft = (dt * xp.fft.rfft(b_in, axis=-1)[:, 1:])[:, freq_mask]

        # Compute inner products over given channels
        inner_prod = 4 * df * xp.abs(((a_fft.conj() * b_fft)) / PSD[:, freq_mask]).sum()

    if use_gpu:
        inner_prod = inner_prod.get()

    return inner_prod


In [90]:
chi2=0
deviation_included=True
evolve_1PA=False
evolve_primary=False
evolve_2PA=False
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

def loglike_calc(m1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_):
    add_args__ = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,\
                dev0p_,dev0e_,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
    waveform_temp=xp.array(superkludge_wave(m1_, m2_, a_, p0_, e0_, xI0, dist, qS_, phiS_, qK, phiK, Phi_phi0_, Phi_theta0, Phi_r0_, *add_args__, dt=dt, T=T,use_gpu=use_gpu))

    diff_inner=inner_product(waveform_true-waveform_temp,waveform_true-waveform_temp,PSD,dt,use_gpu=use_gpu)
    #print(diff_inner)
    return -0.5 * diff_inner*50000

def loglike_calc_snr(m1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_):
    add_args__ = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,\
                dev0p_,dev0e_,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
    waveform_temp=xp.array(superkludge_wave(m1_, m2_, a_, p0_, e0_, xI0, dist, qS_, phiS_, qK, phiK, Phi_phi0_, Phi_theta0, Phi_r0_, *add_args__, dt=dt, T=T))
    diff_inner=inner_prod_without_phase(waveform_temp,waveform_true,PSD,dt,use_gpu=use_gpu)/np.sqrt(inner_prod_without_phase(waveform_temp,waveform_temp,PSD,dt,use_gpu=use_gpu))
    #print(diff_inner)
    return -0.5 * diff_inner


In [91]:
# loglike_calc_snr(np.log(m1), m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_0_p, dev_0_e)

In [92]:
def log_density(params):
    params = np.asarray(params)
    n_samples = params.shape[0] 
    log_likes = np.zeros(n_samples)
    for i in range(n_samples):
        logm1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_ = params[i]
        m1_ = np.exp(logm1_)

        loglike = loglike_calc(m1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_)
        log_likes[i] = loglike 
    return log_likes

n=0.3

logm1lim = [max(0,params_truth_in[0] - n*std[0]), params_truth_in[0] + n*std[0]]
m2lim = [max(0,params_truth_in[1] - n*std[1]), params_truth_in[1] + n*std[1]]
alim = [max(-0.999,params_truth_in[2] - n*std[2]), min(params_truth_in[2] + n*std[2], 0.999)]  # a must be <1
p0lim = [max(0,params_truth_in[3] - n*std[3]), params_truth_in[3] + n*std[3]]
e0lim = [max(0,params_truth_in[4] - n*std[4]), min(1,params_truth_in[4] + n*std[4])]
qSlim = [params_truth_in[5] - n*std[5], params_truth_in[5] + n*std[5]]
phiSlim = [params_truth_in[6] - n*std[6], params_truth_in[6] + n*std[6]]
Phi_phi0lim = [params_truth_in[7] - n*std[7], params_truth_in[7] + n*std[7]]
Phi_r0lim = [params_truth_in[8] - n*std[8], params_truth_in[8] + n*std[8]]
dev0plim = [params_truth_in[9] - n*std[9], params_truth_in[9] + n*std[9]]
dev0elim = [params_truth_in[10] - n*std[10], params_truth_in[10] + n*std[10]]

def prior_transform(u):

    transformed = np.zeros_like(u)
    # Uniform in log for masses
    # m1
    transformed[:, 0] = (logm1lim[1] - logm1lim[0]) * u[:, 0] + logm1lim[0]

    # m2
    transformed[:, 1] = (m2lim[1] - m2lim[0]) * u[:, 1] + m2lim[0]

    # Linear in others 
    # a
    transformed[:, 2] = (alim[1] - alim[0]) * u[:, 2] + alim[0]
    transformed[:, 3] = (p0lim[1] - p0lim[0]) * u[:, 3] + p0lim[0] 
    transformed[:, 4] = (e0lim[1] - e0lim[0]) * u[:, 4] + e0lim[0]
    transformed[:, 5] = (qSlim[1] - qSlim[0]) * u[:, 5] + qSlim[0]
    transformed[:, 6] = (phiSlim[1] - phiSlim[0]) * u[:, 6] + phiSlim[0]
    transformed[:, 7] = (Phi_phi0lim[1] - Phi_phi0lim[0]) * u[:, 7] + Phi_phi0lim[0]
    transformed[:, 8] = (Phi_r0lim[1] - Phi_r0lim[0]) * u[:, 8] + Phi_r0lim[0]
    transformed[:, 9] = (dev0plim[1] - dev0plim[0]) * u[:, 9] + dev0plim[0]
    transformed[:, 10] = (dev0elim[1] - dev0elim[0]) * u[:, 10] + dev0elim[0]

    return transformed

    
def inverse_prior_transform(x):
    
    u = np.zeros_like(x)

    u[:, 0] = (x[:, 0] - logm1lim[0]) / (logm1lim[1] - logm1lim[0])
    u[:, 1] = (x[:, 1] - m2lim[0]) / (m2lim[1] - m2lim[0])
    u[:, 2] = (x[:, 2] - alim[0]) / (alim[1] - alim[0])
    u[:, 3] = (x[:, 3] - p0lim[0]) / (p0lim[1] - p0lim[0])
    u[:, 4] = (x[:, 4] - e0lim[0]) / (e0lim[1] - e0lim[0])
    u[:, 5] = (x[:, 5] - qSlim[0]) / (qSlim[1] - qSlim[0])
    u[:, 6] = (x[:, 6] - phiSlim[0]) / (phiSlim[1] - phiSlim[0])
    u[:, 7] = (x[:, 7] - Phi_phi0lim[0]) / (Phi_phi0lim[1] - Phi_phi0lim[0])
    u[:, 8] = (x[:, 8] - Phi_r0lim[0]) / (Phi_r0lim[1] - Phi_r0lim[0])
    u[:, 9] = (x[:, 9] - dev0plim[0]) / (dev0plim[1] - dev0plim[0])
    u[:, 10] = (x[:, 10] - dev0elim[0]) / (dev0elim[1] - dev0elim[0])
    
    return u

In [93]:

logm1lim = [max(0,params_truth_in[0] - n*std[0]), params_truth_in[0] + n*std[0]]
m2lim = [max(0,params_truth_in[1] - n*std[1]), params_truth_in[1] + n*std[1]]
alim = [max(-0.999,params_truth_in[2] - n*std[2]), min(params_truth_in[2] + n*std[2], 0.999)]  # a must be <1
p0lim = [max(0,params_truth_in[3] - n*std[3]), params_truth_in[3] + n*std[3]]
e0lim = [max(0,params_truth_in[4] - n*std[4]), min(1,params_truth_in[4] + n*std[4])]
qSlim = [params_truth_in[5] - n*std[5], params_truth_in[5] + n*std[5]]
phiSlim = [params_truth_in[6] - n*std[6], params_truth_in[6] + n*std[6]]
Phi_phi0lim = [params_truth_in[7] - n*std[7], params_truth_in[7] + n*std[7]]
Phi_r0lim = [params_truth_in[8] - n*std[8], params_truth_in[8] + n*std[8]]
dev0plim = [params_truth_in[9] - n*std[9], params_truth_in[9] + n*std[9]]
dev0elim = [params_truth_in[10] - n*std[10], params_truth_in[10] + n*std[10]]
ranges=[logm1lim, m2lim, alim, p0lim, e0lim, qSlim, phiSlim, Phi_phi0lim, Phi_r0lim, dev0plim, dev0elim]
ranges

[[np.float64(13.813308893119139), np.float64(13.817712222809408)],
 [np.float64(9.977715816847889), np.float64(10.022284183152111)],
 [np.float64(0.7986015478006661), np.float64(0.801398452199334)],
 [np.float64(7.491797121399476), np.float64(7.508202878600524)],
 [np.float64(0.3993875017683181), np.float64(0.40061249823168193)],
 [np.float64(0.7773419070426192), np.float64(0.7934544197522774)],
 [np.float64(0.9905788663297143), np.float64(1.0094211336702859)],
 [np.float64(0.8609186050450254), np.float64(0.9390813949549747)],
 [np.float64(0.38842104946428874), np.float64(0.4115789505357113)],
 [np.float64(-0.0028669514387040535), np.float64(0.0028669514387040535)],
 [np.float64(-0.003971914917447843), np.float64(0.003971914917447843)]]

In [96]:
# def main():
    
config = SamplerConfig(
    merge_confidence=0.9,          # Coverage prob → Mahalanobis merge radius R_m (higher is more permissive)
    alpha=5000,                    # Use recent samples for weighting
    trail_size=int(2e3),          # Maximum trials per iteration
    boundary_limiting=True,        # Enable boundary constraints
    use_beta=True,                # Use beta correction for boundaries
    integral_num=int(1e5),        # MC samples for beta estimation
    gamma=500,                    # Covariance update frequency
    exclude_scale_z=10,       # No exclusion based on weights
    use_pool=False,               # Set to True for multiprocessing
    # n_pool=4                     # Number of processes (if use_pool=True)
)

ndim = 11
n_seed = 100  # Number of initial processes
init_cov_list = [np.eye(ndim) * 1e-14] * n_seed
savepath = 'paris_manin_t_05_last_chnaged_n_to_broaden_inverse'  # Directory to save results

# Create save directory
os.makedirs(savepath, exist_ok=True)

print(f"Problem dimension: {ndim}")
print(f"Number of processes: {n_seed}")
print(f"Save path: {savepath}")
print(f"Multiprocessing: {config.use_pool}")
# Initialize sampler
print("\nInitializing sampler...")
sampler = Sampler(
    ndim=ndim, 
    n_seed=n_seed,
    log_density_func=log_density,
    init_cov_list=init_cov_list,
    prior_transform=prior_transform,
    config=config
)
   # Prepare initial samples using Latin Hypercube Sampling
print("Preparing LHS samples...")
# sampler.prepare_lhs_samples(lhs_num=int(5e4), batch_size=50)
# x = np.linspace(0.49999, 0.50001, 2000)

# # Generate 99 random points in 11D
# external_lhs_points = np.random.choice(x, size=(99, 11))

# # Add the "true point" [0.5, 0.5, ..., 0.5] as the 100th point
# external_lhs_points = np.vstack([external_lhs_points, [0.5]*11])

#best value got so far
true_point = np.array([0.528488  , 0.49905067, 0.53478167, 0.46723163, 0.48051015,
        0.5031039 , 0.49968549, 0.50062234, 0.49080897, 0.51353855,
        0.51171729])

scatter = 5.0e-8
points = true_point + np.random.randn(99, 11) * scatter
# Add the original point as the 100th row
external_lhs_points = np.vstack([points, true_point])
print("Shape of points array:", external_lhs_points.shape)
external_lhs_log_densities = log_density(prior_transform(external_lhs_points))
print("true points in corrrect space",params_truth_in)
print("external_lhs_log_densities", external_lhs_log_densities)


# external_lhs_points = np.vstack(external_lhs_points)
# external_lhs_log_densities = np.concatenate(external_lhs_log_densities)

sampler.run_sampling(
            num_iterations=int(1e5),
            savepath=savepath,
            print_iter=100,
            external_lhs_points=external_lhs_points,
            external_lhs_log_densities=external_lhs_log_densities,
            stop_dlogZ=0.01
        )
#except Exception as exc:
 #       print(f"[WARN] PARIS sampling failed: {exc}")
    
# Gt results
print("Extracting results...")
samples_, weights_ = sampler.get_samples_with_weights(flatten=True)

# Basic analysis
print(f"\nResults Summary:")
print(f"Total samples: {len(samples_)}")
print(f"Effective sample size: {1/np.sum(weights_  **2):.1f}")

# Weighted statistics
weighted_mean = np.average(samples_, weights=weights_, axis=0)
weighted_cov = np.cov(samples_.T, aweights=weights_)

print(f"\nTrue values: {params_truth_in}")
print(f"Estimated deviation: {weighted_mean}")
print(f"Mean deviation: {np.linalg.norm(weighted_mean - params_truth_in):.6f}")

print(f"\nTrue covariance diagonal: {np.diag(cov)}")
print(f"Estimated covariance diagonal: {np.diag(weighted_cov)}")


Problem dimension: 11
Number of processes: 100
Save path: paris_manin_t_05_last_chnaged_n_to_broaden_inverse
Multiprocessing: False

Initializing sampler...
Preparing LHS samples...
Shape of points array: (100, 11)
true points in corrrect space [13.81551056 10.          0.8         7.5         0.4         0.78539816
  1.          0.9         0.4         0.          0.        ]
external_lhs_log_densities [-258.40158009 -167.14361233 -229.95290393 -208.28507607 -225.10102356
 -229.3428992  -208.0489173  -217.74477723 -230.02019019 -225.80674568
 -236.2587056  -228.95259523 -189.36560393 -179.33093696 -194.5409526
 -217.08508737 -245.24353212 -196.91198611 -195.29200909 -215.36727732
 -224.49493566 -260.02317285 -225.02650588 -271.02675079 -240.33158909
 -220.07697969 -198.30796179 -200.04821028 -292.46569031 -288.46569617
 -246.977222   -214.8553277  -229.15305137 -205.84517424 -211.8911811
 -198.24783271 -220.3774232  -227.2489194  -239.88468336 -261.55067533
 -224.85026166 -202.2605285

Sampling:   0%|          | 0/99999 [00:00<?, ?it/s]

Sampling interrupted by user


Extracting results...

Results Summary:
Total samples: 27039
Effective sample size: 0.0

True values: [13.81551056 10.          0.8         7.5         0.4         0.78539816
  1.          0.9         0.4         0.          0.        ]
Estimated deviation: [1.38156423e+01 9.99976196e+00 8.00101263e-01 7.49943906e+00
 3.99974355e-01 7.85408783e-01 9.99999302e-01 9.00112123e-01
 3.99800334e-01 1.02091787e-04 1.21856281e-04]
Mean deviation: 0.000691

True covariance diagonal: [5.38592010e-05 5.51760910e-03 2.17296506e-05 7.47635748e-04
 4.16837871e-06 7.21147405e-04 9.86197329e-04 1.69706159e-02
 1.48968995e-03 9.13267839e-05 1.75290090e-04]
Estimated covariance diagonal: [1.01132214e-09 3.23728430e-07 4.07952196e-10 1.40377292e-08
 7.82784493e-11 1.44512247e-08 1.86165462e-08 3.34016674e-07
 2.89259127e-08 3.66484731e-09 5.02398854e-09]


In [98]:
best_fit_tn=external_lhs_points[np.argmax(external_lhs_log_densities)]
print(np.max(external_lhs_log_densities))
print(best_fit_tn)
prior_transform(np.array([best_fit_tn]))

-167.14361232506087
[0.52848789 0.49905068 0.5347817  0.46723162 0.48051013 0.50310392
 0.49968542 0.50062227 0.49080892 0.51353855 0.51171722]


array([[1.38156360e+01, 9.99995769e+00, 8.00097281e-01, 7.49946241e+00,
        3.99976125e-01, 7.85448175e-01, 9.99994073e-01, 9.00048638e-01,
        3.99787154e-01, 7.76287206e-05, 9.30796267e-05]])

# array([-0.0309915])

## [ 1.38155486e+01,  9.99979445e+00,  8.00040917e-01,7.49979222e+00,  3.99999718e-01,  7.85405811e-01,9.99994002e-01,  8.99976505e-01,  3.99990972e-01,2.65747370e-05, -6.02634902e-06]

# array([-0.03797392])
## [1.38155436e+01,9.99981186e+00,8.00037769e-01,7.49981068e+00,4.00001137e-01,7.85400514e-01,9.99999135e-01,8.99993267e-01,3.99997760e-01  ,2.08339631e-05, -1.49621831e-05]

# array([-0.03823627])
## [ 1.38155436e+01,  9.99981186e+00,  8.00037769e-01,7.49981068e+00,  4.00001137e-01,  7.85400513e-01,9.99999134e-01,  8.99993271e-01,  3.99997763e-01,2.08333075e-05, -1.49625117e-05]

# Best value (with t = 0.5)
# array([-0.02666697]) 
## [1.38155486e+01,9.99979446e+00,8.00040917e-01,7.49979221e+00,3.99999719e-01,7.85405818e-01,9.99994002e-01,8.99976505e-01,3.99990964e-01,2.65739624e-05,-6.02376376e-06]

# -0.01027893
## [1.38156360e+01, 9.99995768e+00, 8.00097282e-01, 7.49946242e+00,3.99976125e-01, 7.85448171e-01 ,9.99994074e-01 ,9.00048662e-01, 3.99787156e-01 ,7.76271701e-05 ,9.30805528e-05]

# array([-0.01023626])

## [1.38156360e+01, 9.99995768e+00, 8.00097282e-01, 7.49946242e+00,3.99976125e-01, 7.85448172e-01, 9.99994074e-01, 9.00048647e-01,3.99787154e-01, 7.76273556e-05, 9.30805399e-05]

# array([-0.00290299])

## [1.38156360e+01, 9.99995769e+00, 8.00097281e-01, 7.49946241e+00,3.99976125e-01, 7.85448175e-01, 9.99994074e-01, 9.00048644e-01,3.99787155e-01, 7.76287042e-05, 9.30801699e-05]

In [100]:
print(log_density([params_truth_in]))
log_density([[1.38156360e+01, 9.99995769e+00, 8.00097281e-01, 7.49946241e+00,
        3.99976125e-01, 7.85448175e-01, 9.99994073e-01, 9.00048638e-01,
        3.99787154e-01, 7.76287206e-05, 9.30796267e-05]])/50000



[-7071203.49070448]


array([-0.00437625])

In [89]:
inverse_prior_transform(np.array([[1.38156360e+01, 9.99995769e+00, 8.00097281e-01, 7.49946241e+00,3.99976125e-01,
                                    7.85448175e-01, 9.99994074e-01, 9.00048644e-01,3.99787155e-01, 7.76287042e-05, 9.30801699e-05]]))

array([[0.528488  , 0.49905067, 0.53478167, 0.46723163, 0.48051015,
        0.5031039 , 0.49968549, 0.50062234, 0.49080897, 0.51353855,
        0.51171729]])

In [ ]:
import corner

param_ranges= [logm1lim,m2lim,alim,p0lim,e0lim,qSlim,phiSlim,Phi_phi0lim,Phi_r0lim,dev0plim,dev0elim]

# scale ranges → must become list of (min, max) tuples
# param_ranges = [(l[0] * n_scaling, l[1] * n_scaling) for l in param_ranges]
print(param_ranges)

labels = [
    "logm1", "m2", "a", "p0", "e0",
    "qS", "phiS", "Phi_phi0", "Phi_r0",
    "dev0p", "dev0e"
]

fig = corner.corner(
    samples_,
    weights=weights_,
    labels=labels,
    truths=params_truth_in,
    truth_color="red",
    color="green",
    show_titles=True,
    label_kwargs={"fontsize": 10},
    title_kwargs={"fontsize": 12},
    quantiles=[0.16, 0.5, 0.84],
    smooth=True,
    bins=25,
    plot_datapoints=False,
    hist_kwargs={"density": True, "linewidth": 2.5},
    linewidths=2.5,
    fill_contours=True,
    range=param_ranges)

fig.show()


In [ ]:
def summarize(sampler: Sampler) -> None:
    print("Sampler State")
    print("-------------")
    print(f"ndim: {sampler.ndim}")
    print(f"n_proc: {sampler.n_seed}")
    print(f"current_iter: {getattr(sampler, 'current_iter', None)}")
    print(f"savepath: {getattr(sampler, 'savepath', '(unset)')}")
    
state_path = '/home/svu/e1583490/scratch/paris_manin_t_05_last/sampler_state.pkl'
if not os.path.isfile(state_path):
    print(f"Sampler state not found at: {state_path}")
    print("Please run the multimodal example first:")
    print("  python examples/multimodal_example.py")
    raise FileExistsError(f"State file not found: {state_path}")


sampler = Sampler.load_state(state_path)

# Optionally, rebind the functions to the wrappers for clarity
# (unpickling may already have set them to these wrappers).
try:
        sampler.log_density_func_original = log_density
        if hasattr(sampler, "prior_transform") and sampler.prior_transform is not None:
            sampler.prior_transform = prior_transform
        # If prior_transform was set, ensure transformed log-density hook is in place
        if getattr(sampler, "prior_transform", None) is not None:
            sampler.log_density_func = sampler.transformed_log_density_func
        else:
            sampler.log_density_func = sampler.log_density_func_original
        print("Rebound functions after loading state.")
except Exception:
        # Keep going even if attributes differ in older states
        pass

print("Loaded state. Summary before resume:")
summarize(sampler)

    # Continue sampling
print("\nResuming sampling...")
sampler.run_sampling(
num_iterations=int(1e4),
        savepath="paris_manin_t_05_cov14_30k",
        print_iter=200,
        stop_dlogZ=0.01,
    )

print("\nResume completed. Summary after resume:")
summarize(sampler)

    # Optional: quick analysis similar to load_sampler_example
try:
        import numpy as np
        samples, weights = sampler.get_samples_with_weights(flatten=True)
        ess = 1.0 / (weights ** 2).sum()
        wmean = (samples * weights[:, None]).sum(axis=0) / weights.sum()
        print("\nQuick analysis")
        print("--------------")
        print(f"Total samples: {len(samples)}")
        print(f"Effective sample size (ESS): {ess:.1f}")
        print(f"Weighted mean (first 5 dims): {wmean[:5]}")
except Exception as e:
        print(f"Analysis skipped: {e}")


In [ ]:
wmean

In [ ]:
print(f"Weighted mean (first 5 dims): {wmean[:]}")

In [ ]:
import corner

param_ranges= [logm1lim,m2lim,alim,p0lim,e0lim,qSlim,phiSlim,Phi_phi0lim,Phi_r0lim,dev0plim,dev0elim]

# scale ranges → must become list of (min, max) tuples
# param_ranges = [(l[0] * n_scaling, l[1] * n_scaling) for l in param_ranges]
print(param_ranges)
samples_, weights_ = sampler.get_samples_with_weights(flatten=True)
labels = [
    "logm1", "m2", "a", "p0", "e0",
    "qS", "phiS", "Phi_phi0", "Phi_r0",
    "dev0p", "dev0e"
]

fig = corner.corner(
    samples_,
    weights=weights_,
    labels=labels,
    truths=params_truth_in,
    truth_color="red",
    color="green",
    show_titles=True,
    label_kwargs={"fontsize": 10},
    title_kwargs={"fontsize": 12},
    quantiles=[0.16, 0.5, 0.84],
    smooth=True,
    bins=20,
    plot_datapoints=False,
    hist_kwargs={"density": True, "linewidth": 2.5},
    linewidths=2.5,
    fill_contours=True,
    range=param_ranges)

fig.show()


In [ ]:
n=0.01
logm1lim = [max(0,params_truth_in[0] - n*std[0]), params_truth_in[0] + n*std[0]]
logm_range=np.linspace(logm1lim[0],logm1lim[1],1000)
val_m=[]

for i in logm_range:
    loglike=log_density(np.array([[i, m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_0_p, dev_0_e]]))
    val_m.append(loglike)


In [ ]:
plt.plot(logm_range,val_m)
plt.axvline(x=params_truth_in[0],color='red')
plt.xlabel('logm1')
plt.ylabel('loglike')
plt.title('loglike vs logm1 (0.01 std)')

In [ ]:
plt.plot(logm_range,val_m)
plt.axvline(x=qS,color='red')
plt.xlabel('qS')
plt.ylabel('loglike')
plt.title('loglike vs qS (1 std)')

In [ ]:
plt.plot(logm_range,val_m)
plt.axvline(x=0,color='red')
plt.xlabel('dev_0_p')
plt.ylabel('loglike')
plt.title('loglike vs dev_0_p (1 std)')

In [ ]:
plt.plot(logm_range,val_m)
plt.axvline(x=params_truth_in[4],color='red')
plt.xlabel('e0')
plt.ylabel('loglike')
plt.title('loglike vs e0 (1 std)')

In [ ]:
from stableemrifisher.plot import CovEllipsePlot
# pars_list = [m1, m2, p0, e0, dist,qS,phiS,Phi_phi0,Phi_r0,chi2]
# param_names = ['m1','m2','p0','e0','dist','qS','phiS','Phi_phi0','Phi_r0','chi2']
pars_list = params_truth_in
# param_names_plot = [
#     r'$\log{m_1}$', r'$m_2$', r'$a$', r'$p_0$', r'$e_0$',
#     r'$\phi_S$', r'$\Phi_{\phi_0}$', r'$\Phi_{r_0}$',
#     r'dev0_p', r'dev0_e'
# ]
param_names = ['ln(m1)','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']

wave_params = {}
for i in range(len(pars_list)):
    wave_params[param_names[i]] = params_truth_in[i]

ellipse_kwargs = dict(facecolor='None', edgecolor='b', lw=2)
line_kwargs = dict(lw=2, color='b')
covariance= np.linalg.inv(fisher_)
fig, axs = plt.subplots(len(covariance),len(covariance), figsize=(20,20))
fig, axs = CovEllipsePlot(covariance, wave_params=wave_params, param_names=param_names, fig=fig, axs=axs, ellipse_kwargs=ellipse_kwargs, line_kwargs=line_kwargs)
# === ADD TRUE AND BIASED PARAMETER LINES ===
for i in range(len(param_names)):
    # Diagonal subplot (1D)
    ax = axs[i, i]
    ax.axvline(params_truth_in[i], color='k', linestyle='-', lw=2, label='True')
    ax.legend(fontsize=10)


plt.tight_layout()
plt.show()


In [ ]:
from stableemrifisher.plot import CovEllipsePlot
import numpy as np
import matplotlib.pyplot as plt

pars_list = params_truth_in
param_names = ['ln(m1)','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']

# --- build wave_params dict (same as original) ---
wave_params = {param_names[i]: params_truth_in[i] for i in range(len(pars_list))}

ellipse_kwargs = dict(facecolor='None', edgecolor='b', lw=2)
line_kwargs = dict(lw=2, color='b')

# --- create subplots grid ---
fig, axs = plt.subplots(len(covariance), len(covariance), figsize=(20,20))

# --- Fisher covariance ellipse corner plot ---
fig, axs = CovEllipsePlot(
    covariance,
    wave_params=wave_params,
    param_names=param_names,
    fig=fig,
    axs=axs,
    ellipse_kwargs=ellipse_kwargs,
    line_kwargs=line_kwargs
)

# === ADD TRUE PARAMETER LINES (existing behavior) ===
for i in range(len(param_names)):
    ax = axs[i, i]
    ax.axvline(params_truth_in[i], color='k', linestyle='-', lw=2, label='True')
    ax.legend(fontsize=10)

# === ADD 99% CONDITIONAL POSTERIOR LINES ON DIAGONAL ===
for i in range(len(param_names)):

    theta = params_truth_in.copy()  # reference conditioning point
    grid = np.linspace(param_ranges[i][0], param_ranges[i][1], 2000)
    logP = []

    for x in grid:
        theta[i] = x
        lp = log_density(np.array([theta]))
        logP.append(lp[0])


    logP = np.array(logP)
    P = np.exp(logP - logP.max())  # unnormalized conditional posterior
    P /= np.trapezoid(P, grid)         # normalize

    # build CDF and extract 0.5% and 99.5% quantiles
    cdf = np.cumsum(P)
    cdf /= cdf[-1]
    low_99  = grid[np.searchsorted(cdf, 0.005)]
    high_99 = grid[np.searchsorted(cdf, 0.995)]

    # plot the 99% lines on the diagonal subplot
    ax = axs[i, i]
    ax.axvline(low_99,  linestyle="--", lw=2, label="0.5% (conditional)")
    ax.axvline(high_99, linestyle="--", lw=2, label="99.5% (conditional)")
    ax.legend(fontsize=9)

    print(f"{param_names[i]:10s} 99% conditional interval → [{low_99:.6f}, {high_99:.6f}]")

plt.tight_layout()
plt.show()


In [ ]:
from stableemrifisher.plot import CovEllipsePlot
import numpy as np
import matplotlib.pyplot as plt

pars_list = params_truth_in
param_names = ['ln(m1)','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']

# dictionary for fisher ellipse plot (same content as original)
wave_params = {param_names[i]: params_truth_in[i] for i in range(len(pars_list))}

ellipse_kwargs = dict(facecolor='None', edgecolor='b', lw=2)
line_kwargs = dict(lw=2, color='b')

covariance = np.linalg.inv(fisher_)

# create subplot grid
fig, axs = plt.subplots(len(param_names), len(param_names), figsize=(20,20))

# fisher ellipse corner plot
fig, axs = CovEllipsePlot(
    covariance,
    wave_params=wave_params,
    param_names=param_names,
    fig=fig,
    axs=axs,
    ellipse_kwargs=ellipse_kwargs,
    line_kwargs=line_kwargs
)

# === Plot conditional posterior scans for each parameter i, across entire row ===
for i in range(len(param_names)):

    theta = params_truth_in.copy()  # conditioning reference point
    grid = np.linspace(param_ranges[i][0], param_ranges[i][1], 2000)

    logP = []
    for x in grid:
        theta[i] = x
        lp = log_density(np.array([theta]))
        logP.append(lp[0])

    logP = np.array(logP)
    P = np.exp(logP - logP.max())  # conditional posterior density ∝ exp(logP)
    P /= np.trapz(P, grid)         # normalize

    # compute CDF and 99% quantiles (for the diagonal lines)
    cdf = np.cumsum(P)
    cdf /= cdf[-1]
    low99  = grid[np.searchsorted(cdf, 0.005)]
    high99 = grid[np.searchsorted(cdf, 0.995)]

    # --- plot the conditional probability curve on ALL panels in row i ---
    for j in range(len(param_names)):
        ax = axs[i, j]
        ax.plot(grid, P, lw=1.2)

        # On diagonal i,i we also draw the 99% lines + true line
        if i == j:
            ax.axvline(params_truth_in[i], color='k', lw=2, label='true')
            ax.axvline(low99,  linestyle='--', lw=2, label='0.5% cond')
            ax.axvline(high99, linestyle='--', lw=2, label='99.5% cond')
            ax.legend(fontsize=8)

        # label bottom axes only for last row
        if i == len(param_names)-1:
            ax.set_xlabel(param_names[j], fontsize=9)

        # label left axes only for first column
        if j == 0:
            ax.set_ylabel(f"P({param_names[i]} | others)", fontsize=9)

    print(f"{param_names[i]:10s} 99% conditional interval → [{low99:.6f}, {high99:.6f}]")

plt.tight_layout()
plt.show()


In [ ]:
params_truth_in

In [ ]:
[1.37395954e+01,1.02986893e+01,7.62115004e-01,7.50765922e+00,4.17729451e-01,8.56649609e-01,1.09156164e+00,5.84324383e-01,-1.35031994e-03,
-9.89602655e-02,7.60856375e-02]

In [ ]:
from scipy.optimize import minimize
minimize(fun, x0, args=(), method=None, jac=None, hess=None, hessp=None, bounds=None, constraints=(), tol=None, callback=None, options=None)